In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
sys.path.append("/root/limlab01/kaistai/25DFT/QHFlow/src")
import md.scflow_calculator_gpu
from dft_process.dft_process_utils import *

gpu4pyscf is installed.


/root/miniforge3/envs/pyscf-gpu/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO     [dft_process_utils.py:32] >> Source path: /root/limlab01/kaistai/25DFT/QHFlow/src

INFO     [dft_process_utils.py:62] >> Root path: /root/limlab01/kaistai/25DFT/QHFlow

In [ ]:
# data_index = 1 is ethanol
md17_experiment = MD17Experiment(data_index=1)

md17_experiment.set_model_path()

INFO     [setup.py:66] >> Seed: 0

Seed set to 0


ConfigNamespace({'dataset_name': 'water', 'num_train': 500, 'num_valid': 500, 'density_loss': 0.01, 'max_radius': 15.0, 'batch_size': 16, 'train_batch_size': 16, 'valid_batch_size': 16, 'test_batch_size': 16})


INFO     [dft_process_utils.py:202] >> [!] conf.dataset.dataset_name: ethanol

md17
ethanol
MD17_DFT(30000)


In [5]:
md17_experiment.dataset

MD17_DFT(30000)

In [14]:
md17_experiment.dataset[0].pos * BOHR2ANG, md17_experiment.dataset[0].atoms

(tensor([[-0.3349, -0.4840, -0.1223],
         [-0.9672,  0.7590,  0.7194],
         [ 1.1648, -0.3051, -0.5249],
         [-0.1790, -1.4296,  0.4463],
         [-0.8747, -0.6603, -1.1671],
         [-1.9156,  0.5457,  1.0776],
         [-0.1702,  0.8732,  1.4867],
         [-0.9023,  1.6611,  0.2440],
         [ 1.0706,  0.5758, -0.8720]]),
 tensor([[6],
         [6],
         [8],
         [1],
         [1],
         [1],
         [1],
         [1],
         [1]]))

In [15]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import datetime
import pickle

# ===== 디버그 설정 =====
# None이면 전체 샘플 처리, 숫자를 지정하면 해당 개수만 처리
NUM_SAMPLES = None  # 예: 100, 1000 등으로 설정하면 해당 개수만 처리
# ======================

# 에탄올 구조: [C, C, O, H, H, H, H, H, H]
# O는 인덱스 2, H들은 인덱스 3-8
O_idx = 2
H_indices = [3, 4, 5, 6, 7, 8]

# 처리할 샘플 개수 결정
total_samples = len(md17_experiment.dataset)
if NUM_SAMPLES is None:
    num_samples_to_process = total_samples
    print(f"[전체 처리 모드] 총 {total_samples}개 샘플 모두 처리")
else:
    num_samples_to_process = min(NUM_SAMPLES, total_samples)
    print(f"[디버그 모드] {num_samples_to_process}개 샘플만 처리 (전체: {total_samples}개)")

# 모든 샘플에 대해 OH 거리 계산
oh_min_distances = []  # 가장 가까운 OH 거리 (min distance)
oh_all_distances = []  # 모든 O-H 거리 (all distances)
oh_selected_indices = []  # 각 샘플에서 선택된 H 인덱스
oh_distances_by_index = {h_idx: [] for h_idx in H_indices}  # 인덱스별 거리 저장

print(f"\n{num_samples_to_process}개 샘플 처리 중...")
for i in tqdm(range(num_samples_to_process)):
    sample = md17_experiment.dataset[i]
    pos = sample.pos * BOHR2ANG  # Bohr to Angstrom 변환
    atoms = sample.atoms.flatten()
    
    # O 원자 위치
    o_pos = pos[O_idx].numpy()
    
    # 모든 H 원자와의 거리 계산
    h_distances = []
    for h_idx in H_indices:
        h_pos = pos[h_idx].numpy()
        dist = np.linalg.norm(o_pos - h_pos)
        h_distances.append(dist)
        oh_all_distances.append(dist)  # 모든 거리 저장
    
    # 가장 가까운 H가 OH 결합 (일반적으로 ~0.96 Angstrom)
    min_idx = np.argmin(h_distances)
    selected_h_idx = H_indices[min_idx]
    oh_min_dist = h_distances[min_idx]
    
    oh_min_distances.append(oh_min_dist)
    oh_selected_indices.append(selected_h_idx)
    oh_distances_by_index[selected_h_idx].append(oh_min_dist)

oh_min_distances = np.array(oh_min_distances)
oh_all_distances = np.array(oh_all_distances)
oh_selected_indices = np.array(oh_selected_indices)

print(f"\n=== 최소 OH 거리 (Min Distance) 통계 ===")
print(f"  평균: {oh_min_distances.mean():.4f} Å")
print(f"  표준편차: {oh_min_distances.std():.4f} Å")
print(f"  최소값: {oh_min_distances.min():.4f} Å")
print(f"  최대값: {oh_min_distances.max():.4f} Å")
print(f"  중앙값: {np.median(oh_min_distances):.4f} Å")

print(f"\n=== 모든 O-H 거리 (All Distances) 통계 ===")
print(f"  평균: {oh_all_distances.mean():.4f} Å")
print(f"  표준편차: {oh_all_distances.std():.4f} Å")
print(f"  최소값: {oh_all_distances.min():.4f} Å")
print(f"  최대값: {oh_all_distances.max():.4f} Å")
print(f"  중앙값: {np.median(oh_all_distances):.4f} Å")
print(f"  총 거리 개수: {len(oh_all_distances)}개 (샘플 수 × 6개 H)")

print(f"\n=== 선택된 H 인덱스별 통계 ===")
for h_idx in H_indices:
    if len(oh_distances_by_index[h_idx]) > 0:
        dists = np.array(oh_distances_by_index[h_idx])
        count = len(dists)
        percentage = count / len(oh_selected_indices) * 100
        print(f"  H 인덱스 {h_idx}: {count}회 선택 ({percentage:.2f}%)")
        print(f"    평균 거리: {dists.mean():.4f} Å")
        print(f"    표준편차: {dists.std():.4f} Å")

# 데이터 저장
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = '/root/25DFT/QHFlow/_asset/ethanol_draw/plots'
os.makedirs(save_dir, exist_ok=True)

# 거리 데이터를 딕셔너리로 저장
distance_data = {
    'oh_min_distances': oh_min_distances,
    'oh_all_distances': oh_all_distances,
    'oh_selected_indices': oh_selected_indices,
    'oh_distances_by_index': {k: np.array(v) for k, v in oh_distances_by_index.items()},
    'H_indices': H_indices,
    'O_idx': O_idx,
    'num_samples_processed': num_samples_to_process,
    'total_samples': total_samples,
    'NUM_SAMPLES': NUM_SAMPLES  # 디버그 설정값 저장
}

# 파일명에 샘플 개수 정보 포함
if NUM_SAMPLES is None:
    sample_info = f'all{total_samples}'
else:
    sample_info = f'debug{num_samples_to_process}'

save_path = f'{save_dir}/ethanol_OH_distance_data_{sample_info}_{timestamp}.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(distance_data, f)
print(f"\n거리 데이터 저장됨: {save_path}")
print(f"  처리된 샘플: {num_samples_to_process}개 / 전체: {total_samples}개")

# numpy 배열로도 저장
npz_path = f'{save_dir}/ethanol_OH_distance_data_{sample_info}_{timestamp}.npz'
np.savez(npz_path,
         oh_min_distances=oh_min_distances,
         oh_all_distances=oh_all_distances,
         oh_selected_indices=oh_selected_indices,
         num_samples_processed=num_samples_to_process,
         total_samples=total_samples,
         NUM_SAMPLES=NUM_SAMPLES if NUM_SAMPLES is not None else -1,  # None은 -1로 저장
         **{f'oh_distances_idx_{h_idx}': np.array(oh_distances_by_index[h_idx]) 
            for h_idx in H_indices if len(oh_distances_by_index[h_idx]) > 0})
print(f"NumPy 배열 저장됨: {npz_path}")


[전체 처리 모드] 총 30000개 샘플 모두 처리

30000개 샘플 처리 중...


100%|██████████| 30000/30000 [00:41<00:00, 714.96it/s]



=== 최소 OH 거리 (Min Distance) 통계 ===
  평균: 0.9792 Å
  표준편차: 0.0354 Å
  최소값: 0.8694 Å
  최대값: 1.1342 Å
  중앙값: 0.9781 Å

=== 모든 O-H 거리 (All Distances) 통계 ===
  평균: 2.3428 Å
  표준편차: 0.7713 Å
  최소값: 0.8694 Å
  최대값: 3.7786 Å
  중앙값: 2.2752 Å
  총 거리 개수: 180000개 (샘플 수 × 6개 H)

=== 선택된 H 인덱스별 통계 ===
  H 인덱스 8: 30000회 선택 (100.00%)
    평균 거리: 0.9792 Å
    표준편차: 0.0354 Å

거리 데이터 저장됨: /root/25DFT/QHFlow/_asset/ethanol_draw/plots/ethanol_OH_distance_data_all30000_20260107_152654.pkl
  처리된 샘플: 30000개 / 전체: 30000개
NumPy 배열 저장됨: /root/25DFT/QHFlow/_asset/ethanol_draw/plots/ethanol_OH_distance_data_all30000_20260107_152654.npz
